In [4]:
import arcpy
import math

# --- Table paths ---
capacity_density_tbl = r"C:\Users\KyleSteen\Documents\ArcGIS\Projects\Capacity_Density\cap_dens.gdb\Capacity_Density"
analysis_tbl = r"C:\Users\KyleSteen\Documents\ArcGIS\Projects\Capacity_Density\Capacity_Density.gdb\CONUS_Level_1_Analysis_CapacityDensity"

# --- Field names ---
lat_lookup_field = "latitude_degN"
cap_density_field = "cap_density_X_derate_factor"

analysis_lat_field = "Latitude"
area_field = "sq_m_geodesic"
output_field = "Capacity_kWdc"

# --- Add output field if missing ---
#existing_fields = [f.name for f in arcpy.ListFields(analysis_tbl)]
#if output_field not in existing_fields:
#    arcpy.AddField_management(analysis_tbl, output_field, "DOUBLE")

# --- Build lookup dictionary ---
lat_to_capacity = {}
with arcpy.da.SearchCursor(
    capacity_density_tbl,
    [lat_lookup_field, cap_density_field]
) as cursor:
    for lat_bin, cap in cursor:
        lat_to_capacity[int(lat_bin)] = cap

# --- Update analysis table ---
with arcpy.da.UpdateCursor(
    analysis_tbl,
    [analysis_lat_field, area_field, output_field]
) as cursor:
    for lat, area, _ in cursor:

        if lat is None or area is None:
            cursor.updateRow((lat, area, None))
            continue

        # Bin latitude into 2-degree bands
        lat_bin = int(math.floor(lat / 2.0) * 2)

        if lat_bin in lat_to_capacity:
            capacity_kwdc = lat_to_capacity[lat_bin] * area
            cursor.updateRow((lat, area, capacity_kwdc))
        else:
            cursor.updateRow((lat, area, None))

print("Capacity_kWdc calculated using 2-degree latitude bins.")


Capacity_kWdc calculated using 2-degree latitude bins.
